# Notebook 07 — Ensemble Methods and Gradient Boosting

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 07 of 15  
**Time estimate:** 90 minutes

> Gradient boosting wins more Kaggle competitions than any other algorithm.
> For tabular biological data where deep learning isn't yet competitive,
> XGBoost or LightGBM is usually the best non-neural method.

---
## Step 1 — Motivation

Variant effect prediction (CADD, REVEL), drug response prediction (GDSC), and
clinical outcome modeling are dominated by gradient boosted decision trees when
sample sizes are in the hundreds to tens of thousands. Understanding boosting from
first principles — how each tree corrects the errors of all previous trees —
gives intuition for why it works so reliably.

---
## Step 2 — Intuition

**Bagging (Random Forest) vs. Boosting:**
- **Bagging:** Train many trees in parallel on bootstrap samples. Average predictions.
  Reduces variance. Each tree is independently strong.
- **Boosting:** Train trees sequentially. Each new tree corrects the errors of the
  ensemble so far. Reduces bias. Each tree is weak (shallow), but the sequence
  of corrections produces a strong learner.

**AdaBoost (Freund & Schapire 1997):**
- Upweight misclassified samples after each tree
- New tree focuses on the hard examples
- Final prediction: weighted vote of all trees

**Gradient Boosting (Friedman 2001):**
- Generalization: fit each tree to the **negative gradient** of the loss function
- For squared error loss, the negative gradient = residuals → trees fit the residuals
- For any differentiable loss: trees fit the pseudo-residuals
- Regularization: shrinkage (learning rate η), max tree depth, subsample

**XGBoost additions:**
- Second-order Taylor expansion of loss (uses Hessian)
- L1+L2 regularization on leaf weights
- Column (feature) subsampling per tree and per split
- Sparsity-aware split finding (handles missing data natively)

---
## Step 3 — Biological Background

**CADD (Combined Annotation-Dependent Depletion):**
- Scores the deleteriousness of genetic variants
- Input: 60+ genomic features (conservation, regulatory, splicing, protein effect)
- Model: SVM and gradient boosting ensemble trained on simulated variants (negative)
  vs. observed human variants (positive) via evolutionary constraint
- Output: PHRED-scaled score; >10 = top 10% most deleterious; >20 = top 1%

**GDSC (Genomics of Drug Sensitivity in Cancer):**
- Drug response prediction from cancer cell line genomic features
- XGBoost on mutation status, copy number, expression for IC50 prediction

**Stacking (ensemble of ensembles):**
- Level 0: train multiple models (RF, SVM, GBT, logistic regression)
- Level 1: train a meta-model on the out-of-fold predictions of level 0 models
- Common in computational biology challenges (DREAM, Kaggle) where marginal
  gains matter

---
## Step 4 — Mathematical Explanation

**Gradient Boosting algorithm:**

Initialize: $F_0(x) = \arg\min_\gamma \sum_i L(y_i, \gamma)$
(e.g., for squared error: $F_0 = \bar{y}$; for log-loss: $F_0 = \log(\bar{p}/(1-\bar{p}))$)

For $m = 1, \ldots, M$:
1. Compute pseudo-residuals:
   $$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$
2. Fit a tree $h_m$ to the pseudo-residuals $\{(x_i, r_{im})\}$
3. Find optimal leaf weights $\gamma_{jm}$ for each leaf $j$:
   $$\gamma_{jm} = \arg\min_\gamma \sum_{i \in R_{jm}} L(y_i, F_{m-1}(x_i) + \gamma)$$
4. Update: $F_m(x) = F_{m-1}(x) + \eta \sum_j \gamma_{jm} \mathbf{1}[x \in R_{jm}]$

For **squared error** $L = (y-F)^2/2$: pseudo-residuals = residuals $y_i - F_{m-1}(x_i)$.
For **log-loss**: pseudo-residuals = $y_i - \sigma(F_{m-1}(x_i))$ = (label - probability).

**Shrinkage (learning rate $\eta$):** Scales each tree's contribution. Small $\eta$
(e.g., 0.01–0.1) with more trees generally outperforms large $\eta$ with few trees.

In [ ]:
# Step 6 — Python: Gradient boosting from scratch (regression) + sklearn GBT
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.model_selection import cross_val_score, train_test_split

# ---- Gradient Boosting Regressor from scratch ----
class GradBoostRegressorScratch:
    """
    Gradient boosting for squared error loss.
    Pseudo-residuals = residuals = y - F(x)
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None
    
    def fit(self, X, y):
        self.F0 = y.mean()
        F = np.full(len(y), self.F0)
        for m in range(self.n_estimators):
            # Pseudo-residuals for squared error: r = y - F
            residuals = y - F
            # Fit tree to residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            # Update prediction
            F += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self
    
    def predict(self, X):
        F = np.full(len(X), self.F0)
        for tree in self.trees:
            F += self.lr * tree.predict(X)
        return F

# Test on synthetic regression data
rng = np.random.default_rng(42)
X_reg = rng.uniform(0, 6, (200, 1))
y_reg = np.sin(X_reg.ravel()) + rng.normal(0, 0.2, 200)

gb_scratch = GradBoostRegressorScratch(n_estimators=100, learning_rate=0.1, max_depth=3)
gb_scratch.fit(X_reg, y_reg)
y_pred_scratch = gb_scratch.predict(X_reg)
mse_scratch = np.mean((y_pred_scratch - y_reg)**2)
print(f'GBT scratch MSE: {mse_scratch:.4f}')

# ---- Variant pathogenicity classification dataset ----
rng = np.random.default_rng(42)
n = 1000
# Features: conservation(GERP), MAF, functional_score, coding_score, splice_score
X_var = rng.standard_normal((n, 5))
# True rule: pathogenic if (GERP > 1 AND functional > 0.5) OR splice > 1.5
y_var = ((X_var[:, 0] > 1) & (X_var[:, 2] > 0.5)) | (X_var[:, 4] > 1.5)
y_var = y_var.astype(int)
print(f'\nVariant dataset: {n} variants, {y_var.mean():.1%} pathogenic')

X_tr, X_te, y_tr, y_te = train_test_split(X_var, y_var, test_size=0.2, random_state=42, stratify=y_var)

# Compare AdaBoost, GBT, and XGBoost-style
from sklearn.ensemble import RandomForestClassifier
feat_names = ['GERP', 'MAF', 'Functional', 'Coding', 'Splice']

results = {}
for name, clf in [
    ('AdaBoost (50 trees)', AdaBoostClassifier(n_estimators=50, random_state=42)),
    ('GBT (100 trees, lr=0.1)', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                              max_depth=3, random_state=42)),
    ('Random Forest (100 trees)', RandomForestClassifier(n_estimators=100, random_state=42)),
]:
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=5, scoring='roc_auc')
    clf.fit(X_tr, y_tr)
    te_auc = cross_val_score(clf, X_te, y_te, cv=2, scoring='roc_auc').mean()  # rough
    results[name] = {'clf': clf, 'cv': cv_scores}
    print(f'{name}: CV AUROC={cv_scores.mean():.3f}±{cv_scores.std():.3f}')

# Feature importance from GBT
gbt = results['GBT (100 trees, lr=0.1)']['clf']
print('\nGBT feature importances:')
for f, imp in sorted(zip(feat_names, gbt.feature_importances_), key=lambda x:-x[1]):
    print(f'  {f}: {imp:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: GBT regression fit (scratch)
ax = axes[0]
x_plot = np.linspace(0, 6, 200).reshape(-1,1)
y_true_plot = np.sin(x_plot.ravel())
ax.scatter(X_reg, y_reg, alpha=0.3, s=15, color='grey', label='Data')
ax.plot(x_plot, y_true_plot, 'k', lw=2, label='True function')
ax.plot(x_plot, gb_scratch.predict(x_plot), 'tomato', lw=2, label='GBT (scratch)')

# Also show predictions at different training stages
for m in [1, 5, 20, 100]:
    gb_tmp = GradBoostRegressorScratch(n_estimators=m, learning_rate=0.1, max_depth=3)
    gb_tmp.fit(X_reg, y_reg)
    if m in [5, 100]:
        ax.plot(x_plot, gb_tmp.predict(x_plot), lw=1, alpha=0.6, label=f'm={m}')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('A. GBT regression from scratch\n(boosting steps = progressive correction)')
ax.legend(fontsize=7)

# Panel B: Training loss vs. boosting rounds
ax = axes[1]
gbt_full = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                         max_depth=3, random_state=42)
gbt_full.fit(X_tr, y_tr)
train_deviance = gbt_full.train_score_
test_deviance = []
X_te_full = X_te
for i, pred in enumerate(gbt_full.staged_predict_proba(X_te_full)):
    from sklearn.metrics import log_loss
    test_deviance.append(log_loss(y_te, pred))
ax.plot(train_deviance, 'steelblue', lw=1.5, label='Train deviance')
ax.plot(test_deviance, 'tomato', lw=1.5, label='Test deviance')
ax.axvline(np.argmin(test_deviance), color='grey', ls='--', lw=0.8,
             label=f'Best n={np.argmin(test_deviance)+1}')
ax.set_xlabel('Boosting iterations')
ax.set_ylabel('Deviance (log-loss)')
ax.set_title('B. Train vs. test deviance\n(early stopping opportunity)')
ax.legend(fontsize=8)

# Panel C: Feature importance comparison
ax = axes[2]
rf_fi = results['Random Forest (100 trees)']['clf'].feature_importances_
gbt_fi = gbt.feature_importances_
width = 0.35
x_pos = np.arange(len(feat_names))
ax.bar(x_pos - width/2, rf_fi, width, label='Random Forest', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, gbt_fi, width, label='GBT', color='tomato', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(feat_names, rotation=20, ha='right')
ax.set_ylabel('Feature importance')
ax.set_title('C. Feature importance: RF vs. GBT\n(both should rank Splice, GERP highest)')
ax.legend(fontsize=8)

plt.suptitle('Module 13 NB07: Ensemble Methods and Gradient Boosting', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gradient_boosting.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Extend `GradBoostRegressorScratch` to binary classification using log-loss.
   Pseudo-residuals for log-loss = $y_i - \sigma(F(x_i))$.
2. Show early stopping: monitor test deviance and stop training at the optimal
   number of trees. How much does this improve generalization?
3. Implement AdaBoost from scratch: weighted sampling, tree weights $\alpha_m$,
   final weighted vote. Verify on the variant dataset.
4. What is the relationship between gradient boosting and Newton's method for
   optimization? Why does XGBoost use second-order terms?

---
## Step 10 — Quiz

1. What is the key difference between bagging and boosting?
2. In gradient boosting for squared error, what are the pseudo-residuals?
3. What does the learning rate $\eta$ control in gradient boosting?
4. What is early stopping in gradient boosting, and why is it used?
5. Name two regularization techniques specific to XGBoost.

---
## Step 12 — Reflection

> *[In one paragraph: explain why gradient boosting reduces bias while random forests
> reduce variance. In what type of dataset (large n vs. small n, noisy vs. clean)
> would you prefer each?]*

---
*Next: `08_cross_validation_and_model_selection.ipynb`*